<a href="https://colab.research.google.com/github/SiyumiJayawardhane/sl-edu-reform-voices/blob/main/Solution_2410215.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multilingual NLP Analysis of Public YouTube Discussion on Sri Lankan School Education Reforms
**Student ID:** 2410215 | **Module:** CM4608 Natural Language Processing

This notebook collects public YouTube comments about the Sri Lankan school education reforms (English, Sinhala and Tamil) and analyses them with NLP: data collection and EDA (Task 1), preprocessing and tokenisation (Task 2), theme discovery (Task 3), classification (Task 4), structured extraction (Task 5), and evidence-grounded summaries with an ethics discussion (Task 6).

### 0.1 Reproducibility
A fixed random seed is used for every random step, and package versions are printed so the results can be reproduced.

## 1. Task 1 — Video discovery, comment collection and initial exploration

### 1.1 Environment set-up and API connection test

**Purpose:** Connect to the YouTube Data API v3 and confirm that the API key works before collecting any data.

**Approach:**
- Use google-api-python-client, Google’s official Python client for its APIs.
- Read saved API Key Secrets from colab
- The key is restricted in Google Cloud Console to the YouTube Data API v3 only. No OAuth or user login is used, because only public comments are collected
- Run a single test search.

1. Install the library

In [7]:
!pip install -q google-api-python-client

from google.colab import userdata
from googleapiclient.discovery import build
import sys, random, importlib.metadata as md
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
for pkg in ["numpy", "pandas", "google-api-python-client"]:
    print(f"{pkg}: {md.version(pkg)}")

Python: 3.13.18
numpy: 2.4.6
pandas: 2.2.3
google-api-python-client: 2.201.0


2. Read API Key

In [8]:
API_KEY = userdata.get("YOUTUBE_API_KEY")

3. Create the YouTube Client

In [9]:
yt = build("youtube", "v3", developerKey=API_KEY)

In [11]:
# Connection test: one search
# Ranking changes over time
r = yt.search().list(q="Sri Lanka education reform", part="snippet",
                     type="video", maxResults=3, regionCode="LK").execute()
for item in r["items"]:
    print(item["id"]["videoId"], "|", item["snippet"]["channelTitle"], "|", item["snippet"]["title"])

aUKv__hANms | Manoj Dhananjaya | New Education Reforms 2026 Sri Lanka | 2026 Education Reforms | GPA System for Schools | OL 2026
xzC_81piivo | Life with Tishi in SL | Major education reforms in Sri Lanka
76QErEQG89k | Ada Derana | New education reforms set for Grades 2 and 6 next year; Cabinet gives nod for 2027 implementation


QQGkW9hedpU | Betasoft | Grade 6 – New Education Reform 2026 – Subjects, Modules and Evaluation (in Tamil) |
| Mzt_IWl0mCE | Life with Tishi in SL | 1998 Education reforms and 2007 curriculum reforms in Sri Lanka |
| 1vF1RS7cMm0 | SCIENCE ATOM | அதிரடி மாற்றம்! இலங்கையில் 2026 முதல் புதிய கல்வி முறை – Sri Lanka Education Reforms 2026 |

**Output interpretation Oct 26, 2026:** The test returned three videos from three channels, confirming the key and client work. Two observations shaped the collection design: (1) even an English query returned Tamil-language videos, so the corpus will be multilingual; (2) one result concerned the 1998/2007 reforms, so searches are limited by publication date and videos are screened for relevance.

4.Storage


In [12]:
from google.colab import drive
import os

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/NLP_sl-edu-reform-voices"
RAW_DIR  = f"{PROJECT_DIR}/data/raw"        # one file per video (raw API data)
PROC_DIR = f"{PROJECT_DIR}/data/processed"  # combined datasets
DOCS_DIR = f"{PROJECT_DIR}/docs"            # search log + video lists (reproducibility)

for d in (RAW_DIR, PROC_DIR, DOCS_DIR):
    os.makedirs(d, exist_ok=True)           # create folders if missing

print("Saving data under:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving data under: /content/drive/MyDrive/NLP_sl-edu-reform-voices


### 1.2 Collection configuration and logging

**Purpose:** Record the metadata required by Task 1(a) for reproducibility: every search query, the exact date and time it was run, the number of results, and the API quota used. All timestamps are in UTC so they are unambiguous.

In [6]:
import datetime, csv, json, time
from googleapiclient.errors import HttpError

REGION = "LK"                               # rank results as seen from Sri Lanka
PUBLISHED_AFTER = "2024-01-01T00:00:00Z"    # only recent reform discussion
COLLECTION_DATE = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d")
LOG_FILE = f"{DOCS_DIR}/search_log.csv"
QUOTA_USED = 0                              # units used in this Colab session

def now_utc():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")

def log_call(call_type, label, n_results, units):
    """Write one row per API call: when, what, how many results, quota cost."""
    global QUOTA_USED
    QUOTA_USED += units
    new_file = not os.path.exists(LOG_FILE)
    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(["timestamp_utc", "call_type", "query_or_id",
                        "n_results", "quota_units", "session_quota_total"])
        w.writerow([now_utc(), call_type, label, n_results, units, QUOTA_USED])

def api_call(request, call_type, label, units, retries=3):
    """Run an API request safely: log it, retry temporary errors, skip bad videos."""
    for attempt in range(retries):
        try:
            resp = request.execute()
            log_call(call_type, label, len(resp.get("items", [])), units)
            return resp
        except HttpError as e:
            try:
                reason = json.loads(e.content)["error"]["errors"][0]["reason"]
            except Exception:
                reason = str(e.resp.status)
            log_call(f"{call_type}_ERROR", f"{label} ({reason})", 0, units)
            if reason == "quotaExceeded":
                raise RuntimeError("Daily quota used up. Continue after ~1:30 pm Sri Lanka time.")
            if e.resp.status in (500, 503) and attempt < retries - 1:
                time.sleep(2 ** attempt)        # wait 1s, 2s... then retry
                continue
            print(f"   skipped {label}: {reason}")   # e.g. commentsDisabled
            return None

print("Collection date (UTC):", COLLECTION_DATE)

Collection date (UTC): 2026-11-15


**Output interpretation Nov 15, 2026:** All timestamps are UTC. `PUBLISHED_AFTER` limits the search to videos from 2024 onwards because the connection test showed that older reforms (1998/2007) also appear in results

### 1.3 Video discovery by keyword search
Videos are found automatically with `search.list` (up to 50 results per query, published after 2024-01-01, ranked for Sri Lanka).

**Query design:**
- **Three languages** (English, Sinhala, Tamil) so that the corpus reflects all three language communities.
- **Initial queries** covered the general topic ("education reform", "අධ්‍යාපන ප්‍රතිසංස්කරණ", "கல்வி சீர்திருத்தம்").
- **Refined queries:** after inspecting real reform videos, terms used by creators were added: "modules", "2027" (some videos refer to a 2027 rollout), "6 ශ්‍රේණිය" (Grade 6) and "1 සිට 13 දක්වා" (Grades 1–13).
- **Tamil and English queries include "Sri Lanka" / "இலங்கை"**, because generic Tamil queries return Indian (Tamil Nadu) education videos.
- **Politician names were not used** as queries, to avoid biasing the corpus towards political rather than educational discussion.

In [13]:
SEARCH_QUERIES = [
    # English
    "Sri Lanka education reform",
    "Sri Lanka education reform 2026",
    "Sri Lanka education reform 2027",
    "Sri Lanka education reform modules",
    "Sri Lanka new curriculum grade 6",
    # Sinhala
    "අධ්‍යාපන ප්‍රතිසංස්කරණ",                  # education reforms
    "නව අධ්‍යාපන ප්‍රතිසංස්කරණ 2026",           # new education reforms 2026
    "අධ්‍යාපන ප්‍රතිසංස්කරණ 2027",              # education reforms 2027
    "නව අධ්‍යාපන ප්‍රතිසංස්කරණ modules",        # new reforms modules
    "6 ශ්‍රේණිය මොඩියුල",                      # grade 6 modules
    "1 සිට 13 දක්වා අධ්‍යාපන ප්‍රතිසංස්කරණ",                # education reform from grade 1 to 13
    # Tamil
    "இலங்கை கல்வி சீர்திருத்தம்",               # Sri Lanka education reform
    "இலங்கை கல்வி சீர்திருத்தம் 2026",          # ... 2026
    "இலங்கை புதிய கல்வி சீர்திருத்தம் தரம் 6",   # Sri Lanka new reform grade 6
]

found = {}   # video_id -> first query that found it
for q in SEARCH_QUERIES:
    r = api_call(yt.search().list(q=q, part="snippet", type="video",
                                  regionCode=REGION, publishedAfter=PUBLISHED_AFTER,
                                  maxResults=50, order="relevance"),
                 "search", q, 100)
    if r is None:
        continue
    new = sum(it["id"]["videoId"] not in found for it in r["items"])
    for it in r["items"]:
        found.setdefault(it["id"]["videoId"], q)
    print(f"{len(r['items']):3} results ({new:2} new) | {q}")

print(len(found), "unique candidate videos | quota used:", QUOTA_USED)

 50 results (50 new) | Sri Lanka education reform
 50 results (28 new) | Sri Lanka education reform 2026
 50 results (28 new) | Sri Lanka education reform 2027
 50 results (18 new) | Sri Lanka education reform modules
 50 results (36 new) | Sri Lanka new curriculum grade 6
 50 results (36 new) | අධ්‍යාපන ප්‍රතිසංස්කරණ
 50 results (14 new) | නව අධ්‍යාපන ප්‍රතිසංස්කරණ 2026
 50 results (25 new) | අධ්‍යාපන ප්‍රතිසංස්කරණ 2027
 50 results ( 2 new) | නව අධ්‍යාපන ප්‍රතිසංස්කරණ modules
 50 results (36 new) | 6 ශ්‍රේණිය මොඩියුල
 50 results (41 new) | 1 සිට 13 දක්වා අධ්‍යාපන ප්‍රතිසංස්කරණ
 50 results (46 new) | இலங்கை கல்வி சீர்திருத்தம்
 50 results (12 new) | இலங்கை கல்வி சீர்திருத்தம் 2026
 24 results ( 5 new) | இலங்கை புதிய கல்வி சீர்திருத்தம் தரம் 6
377 unique candidate videos | quota used: 1400


**Output interpretation (15 Nov 2026):** The 14 queries returned 674 results, which reduced to **377 unique candidate videos** (1,400 quota units).

- **The general queries found the most new videos:** "Sri Lanka education reform" (50 new), "இலங்கை கல்வி சீர்திருத்தம்" (46) and "1 සිට 13 දක්වා අධ්‍යාපන ප්‍රතිසංස්කරණ" (41). Each language therefore reaches a different set of videos, which supports searching in all three languages.
- **The refined Sinhala terms were useful:** "6 ශ්‍රේණිය මොඩියුල" (36 new) and "1 සිට 13 දක්වා" (41 new) found many videos that the general queries missed, which justifies adding creators' own terms.
- **Some queries overlap heavily:** "නව අධ්‍යාපන ප්‍රතිසංස්කරණ modules" added only 2 new videos, and the 2026 variants added 12–14, showing diminishing returns from adding more similar queries.
- **There is less Tamil content:** the Tamil Grade 6 query returned only 24 results (5 new), suggesting less Tamil-language video coverage of the reform. Tamil may therefore be under-represented in the corpus.
- The 377 candidates still include off-topic videos, so they are screened in 1.4.